In [ ]:
"""
Salary Prediction using Linear Regression
Week 9 - Real-world Application

Predicting salaries based on years of experience
"""

# ============================================================================
# IMPORTS
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import sys
sys.path.append('../src')
from models import LinearRegressionScratch, calculate_metrics
from preprocessing import FeatureScaler

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
np.random.seed(42)


# ============================================================================
# PART 1: GENERATE REALISTIC SALARY DATA
# ============================================================================

print("="*70)
print("SALARY PREDICTION PROJECT")
print("="*70)

# Generate synthetic salary data
np.random.seed(42)
n_samples = 300

# Years of experience (0-20 years)
years_exp = np.random.uniform(0, 20, n_samples)

# Base salary model: Salary = 30000 + 3500*years + noise
# Adding non-linearity for realism
base_salary = 30000
salary_per_year = 3500
experience_factor = years_exp + 0.5 * (years_exp ** 1.2)  # Non-linear growth
noise = np.random.normal(0, 5000, n_samples)

salary = base_salary + salary_per_year * experience_factor + noise

# Create DataFrame
df = pd.DataFrame({
    'YearsExperience': years_exp,
    'Salary': salary
})

# Round values
df['YearsExperience'] = df['YearsExperience'].round(1)
df['Salary'] = df['Salary'].round(0)

print("\n Dataset Overview")
print("-" * 70)
print(f"Number of samples: {len(df)}")
print(f"\nFirst 10 rows:")
print(df.head(10))
print(f"\n Statistical Summary:")
print(df.describe())

# Save to CSV
df.to_csv('../data/processed/salary_data.csv', index=False)
print("\n Data saved to '../data/processed/salary_data.csv'")


# ============================================================================
# PART 2: EXPLORATORY DATA ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("EXPLORATORY DATA ANALYSIS")
print("="*70)

# Check for missing values
print(f"\n Missing values:")
print(df.isnull().sum())

# Correlation analysis
correlation = df['YearsExperience'].corr(df['Salary'])
print(f"\n Correlation between Experience and Salary: {correlation:.4f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Scatter plot
ax1 = axes[0, 0]
ax1.scatter(df['YearsExperience'], df['Salary'], alpha=0.6, s=50, edgecolors='black')
ax1.set_xlabel('Years of Experience', fontsize=12)
ax1.set_ylabel('Salary ($)', fontsize=12)
ax1.set_title('Salary vs Years of Experience', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: Distribution of Years Experience
ax2 = axes[0, 1]
ax2.hist(df['YearsExperience'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
ax2.set_xlabel('Years of Experience', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Experience', fontsize=14, fontweight='bold')
ax2.axvline(df['YearsExperience'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {df["YearsExperience"].mean():.1f}')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Distribution of Salaries
ax3 = axes[1, 0]
ax3.hist(df['Salary'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
ax3.set_xlabel('Salary ($)', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('Distribution of Salaries', fontsize=14, fontweight='bold')
ax3.axvline(df['Salary'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: ${df["Salary"].mean():.0f}')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Box plots
ax4 = axes[1, 1]
box_data = [df['YearsExperience'], df['Salary'] / 1000]  # Salary in thousands
bp = ax4.boxplot(box_data, labels=['Years Exp', 'Salary (K$)'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
ax4.set_ylabel('Value', fontsize=12)
ax4.set_title('Box Plot Analysis', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ============================================================================
# PART 3: PREPARE DATA
# ============================================================================

print("\n" + "="*70)
print("DATA PREPARATION")
print("="*70)

# Prepare features and target
X = df[['YearsExperience']].values
y = df['Salary'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n Data Split:")
print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# Optional: Scale features (good practice)
scaler = FeatureScaler(method='standard')
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================================
# PART 4: TRAIN MODELS
# ============================================================================

print("\n" + "="*70)
print("MODEL TRAINING")
print("="*70)

# Model 1: Custom Implementation
print("\n🔧 Training Custom Linear Regression...")
model_custom = LinearRegressionScratch(learning_rate=0.01, n_iterations=2000)
model_custom.fit(X_train_scaled, y_train, verbose=False)

print(f"✓ Custom Model Trained")
print(f"  Weight (slope): ${model_custom.weights[0]:.2f} per year")
print(f"  Bias (intercept): ${model_custom.bias:.2f}")

# Model 2: Scikit-Learn
print("\n🔧 Training Scikit-Learn Linear Regression...")
model_sklearn = LinearRegression()
model_sklearn.fit(X_train_scaled, y_train)

print(f"✓ Sklearn Model Trained")
print(f"  Weight (slope): ${model_sklearn.coef_[0]:.2f} per year")
print(f"  Bias (intercept): ${model_sklearn.intercept_:.2f}")


# ============================================================================
# PART 5: MODEL EVALUATION
# ============================================================================

print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

# Make predictions
y_pred_custom = model_custom.predict(X_test_scaled)
y_pred_sklearn = model_sklearn.predict(X_test_scaled)

# Calculate metrics
metrics_custom = calculate_metrics(y_test, y_pred_custom)
metrics_sklearn = calculate_metrics(y_test, y_pred_sklearn)

# Display results
print("\n Custom Model Performance:")
print(f"  MSE: ${metrics_custom['MSE']:.2f}")
print(f"  RMSE: ${metrics_custom['RMSE']:.2f}")
print(f"  MAE: ${metrics_custom['MAE']:.2f}")
print(f"  R² Score: {metrics_custom['R2']:.4f}")

print("\n Scikit-Learn Model Performance:")
print(f"  MSE: ${metrics_sklearn['MSE']:.2f}")
print(f"  RMSE: ${metrics_sklearn['RMSE']:.2f}")
print(f"  MAE: ${metrics_sklearn['MAE']:.2f}")
print(f"  R² Score: {metrics_sklearn['R2']:.4f}")

# Interpretation
print("\n Model Interpretation:")
print(f"  On average, salary increases by ${abs(model_sklearn.coef_[0]):.2f} per year of experience")
print(f"  Starting salary (0 years exp): ${model_sklearn.intercept_:.2f}")
print(f"  Model explains {metrics_sklearn['R2']*100:.2f}% of salary variance")


# ============================================================================
# PART 6: VISUALIZE RESULTS
# ============================================================================

print("\n" + "="*70)
print("VISUALIZATION")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Training Data with Regression Line
ax1 = axes[0, 0]
ax1.scatter(X_train, y_train, alpha=0.5, s=40, label='Training Data', color='blue')
ax1.scatter(X_test, y_test, alpha=0.5, s=40, label='Test Data', color='orange')

# Create line for predictions
X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
X_line_scaled = scaler.transform(X_line)
y_line = model_sklearn.predict(X_line_scaled)

ax1.plot(X_line, y_line, 'r-', linewidth=3, label='Regression Line')
ax1.set_xlabel('Years of Experience', fontsize=11)
ax1.set_ylabel('Salary ($)', fontsize=11)
ax1.set_title('Salary Prediction Model', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Predictions vs Actual
ax2 = axes[0, 1]
ax2.scatter(y_test, y_pred_sklearn, alpha=0.6, s=50, edgecolors='black')
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax2.set_xlabel('Actual Salary ($)', fontsize=11)
ax2.set_ylabel('Predicted Salary ($)', fontsize=11)
ax2.set_title('Predictions vs Actual', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Residuals
ax3 = axes[0, 2]
residuals = y_test - y_pred_sklearn
ax3.scatter(y_pred_sklearn, residuals, alpha=0.6, s=50, edgecolors='black')
ax3.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax3.set_xlabel('Predicted Salary ($)', fontsize=11)
ax3.set_ylabel('Residuals ($)', fontsize=11)
ax3.set_title('Residual Plot', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Residuals Distribution
ax4 = axes[1, 0]
ax4.hist(residuals, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
ax4.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax4.set_xlabel('Residuals ($)', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Residuals Distribution', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Plot 5: Error by Experience Level
ax5 = axes[1, 1]
abs_errors = np.abs(residuals)
ax5.scatter(X_test, abs_errors, alpha=0.6, s=50, edgecolors='black')
ax5.set_xlabel('Years of Experience', fontsize=11)
ax5.set_ylabel('Absolute Error ($)', fontsize=11)
ax5.set_title('Prediction Error by Experience', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# Plot 6: Convergence (Custom Model)
ax6 = axes[1, 2]
ax6.plot(model_custom.cost_history, linewidth=2, color='green')
ax6.set_xlabel('Iteration', fontsize=11)
ax6.set_ylabel('Cost (MSE)', fontsize=11)
ax6.set_title('Cost Function Convergence', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================================
# PART 7: MAKE PREDICTIONS FOR NEW DATA
# ============================================================================

print("\n" + "="*70)
print("SALARY PREDICTIONS FOR DIFFERENT EXPERIENCE LEVELS")
print("="*70)

# Create prediction scenarios
experience_levels = np.array([0, 1, 3, 5, 7, 10, 15, 20]).reshape(-1, 1)
experience_scaled = scaler.transform(experience_levels)
predicted_salaries = model_sklearn.predict(experience_scaled)

# Display predictions
print("\n Predicted Salaries:")
print("-" * 50)
for exp, sal in zip(experience_levels.flatten(), predicted_salaries):
    print(f"  {exp:2.0f} years experience: ${sal:,.0f}")

# Create visualization
fig, ax = plt.subplots(figsize=(12, 7))

# Plot all data
ax.scatter(X, y, alpha=0.3, s=30, label='Actual Data', color='lightgray')

# Plot prediction line
X_line = np.linspace(0, 20, 200).reshape(-1, 1)
X_line_scaled = scaler.transform(X_line)
y_line = model_sklearn.predict(X_line_scaled)
ax.plot(X_line, y_line, 'b-', linewidth=3, label='Prediction Model', alpha=0.8)

# Highlight predictions
ax.scatter(experience_levels, predicted_salaries, s=200, c='red', 
           marker='*', edgecolors='black', linewidths=2, 
           label='Predictions', zorder=5)

# Add annotations for key predictions
for exp, sal in zip(experience_levels[::2].flatten(), predicted_salaries[::2]):
    ax.annotate(f'${sal:,.0f}', 
                xy=(exp, sal), 
                xytext=(exp + 0.5, sal + 5000),
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

ax.set_xlabel('Years of Experience', fontsize=13)
ax.set_ylabel('Salary ($)', fontsize=13)
ax.set_title('Salary Prediction Model with Key Predictions', 
             fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================================
# PART 8: MODEL INSIGHTS AND BUSINESS RECOMMENDATIONS
# ============================================================================

print("\n" + "="*70)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("="*70)

# Calculate salary ranges
entry_level = model_sklearn.predict(scaler.transform([[0]]))[0]
mid_level = model_sklearn.predict(scaler.transform([[5]]))[0]
senior_level = model_sklearn.predict(scaler.transform([[10]]))[0]
expert_level = model_sklearn.predict(scaler.transform([[15]]))[0]

print("\n Salary Benchmarks:")
print(f"  Entry Level (0 years): ${entry_level:,.0f}")
print(f"  Mid Level (5 years): ${mid_level:,.0f}")
print(f"  Senior Level (10 years): ${senior_level:,.0f}")
print(f"  Expert Level (15 years): ${expert_level:,.0f}")

print("\n Key Insights:")
print(f"  1. Average salary increase per year: ${abs(model_sklearn.coef_[0]):.2f}")
print(f"  2. Starting salary: ${entry_level:,.0f}")
print(f"  3. Model accuracy (R²): {metrics_sklearn['R2']*100:.1f}%")
print(f"  4. Typical prediction error: ±${metrics_sklearn['RMSE']:,.0f}")

print("\n Recommendations:")
print("  1. Use this model for initial salary negotiations")
print("  2. Consider adding factors: education, location, industry")
print("  3. Update model quarterly with new data")
print("  4. Validate predictions with market research")

print("\n Salary Prediction Project Completed Successfully!")
print("="*70)